# DeepSurv applied on signature covariates

In [1]:
import types
import sys
from numbers import Real, Integral

# Create a fake module to emulate 'sklearn.utils._param_validation'
# (used by skglm in newer versions of scikit-learn, >=1.3)
param_validation = types.ModuleType("sklearn.utils._param_validation")

# Define a minimal replacement for Interval used in _parameter_constraints
class Interval:
    def __init__(self, dtype, left, right, closed="neither"):
        self.dtype = dtype
        self.left = left
        self.right = right
        self.closed = closed

# Define a minimal replacement for StrOptions used in _parameter_constraints
class StrOptions:
    def __init__(self, options):
        self.options = set(options)

# Add the custom classes to the fake module
param_validation.Interval = Interval
param_validation.StrOptions = StrOptions

# Inject the fake module into sys.modules before skglm is imported
# This prevents skglm from raising an ImportError if sklearn < 1.3
sys.modules["sklearn.utils._param_validation"] = param_validation

In [2]:
import pandas as pd
import torch
import numpy as np
import time
import warnings
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
import os

# Add the src directory to the Python path
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
src_path = os.path.abspath(os.path.join(notebook_dir, '..', 'src/multisigbert'))
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Now import our custom modules
from _utils import *
from descriptive_stats_pkg import *
from compression_pkg import *
from survival_analysis_pkg import *
from metrics_plot_results_pkg import *
from other_models import *

/Applications/anaconda3/envs/multisigbert-env/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
df_OG = pd.read_csv("../data/df_study_real_L36_w12.csv")
df_all = convert_date_columns(df_OG)
seq_struc_col = ["PO", "KAR", "TA", "PL"]

In [5]:
df_OG_interp, structured_columns_out = interpolate_structured_sequences(
    df_OG,
    structured_columns=seq_struc_col,
    patient_id_col='ID',
    time_col = 'date_creation')

Number of patients after interpolation: 2527
Number of structured components: 4


In [6]:
df_all = convert_embeddings_column(df_OG_interp)

Converting embeddings: 100%|████████████| 19448/19448 [00:03<00:00, 6436.23it/s]


In [7]:
# Train-test construction
df_train_new_OG, test_groups = make_train_test(df_all)

In [8]:
k_comp=25

# PCA compression
_, R_comp = pca_compression(df_train_new_OG, k_comp, verbose=True)

Compression dimension (bar_p): 25
Explained variance ratio: 97.0109%


In [9]:
df_train_new_OG = apply_linear_projection(df_train_new_OG, R_comp, var_embd="embeddings")
Xt_train, y_train, features_name, nbr_sig, nbr_levy, id_list, df_study_train = prep_signature_cox(df_train_new_OG, var_struct_seq_list_OG=seq_struc_col)

Timestamps normalized for each patient to the [0, 1] range.
Path dimension: 30
Signature components (order 2): 930
Number of events (deaths): 807 out of 1263 (63.90%)


```python
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=1e-4)

sig_cols = [c for c in df_study_train.columns if c.startswith("sig_")]

X_sig = selector.fit_transform(df_study_train[sig_cols])

kept_cols = [col for col, keep in zip(sig_cols, selector.get_support()) if keep]

df_study_train = df_study_train[[c for c in df_study_train.columns 
                                 if not c.startswith("sig_")]]

df_study_train = pd.concat(
    [df_study_train.reset_index(drop=True),
     pd.DataFrame(X_sig, columns=kept_cols)],
    axis=1
)
```

In [10]:
# ============================================================
# DeepSurv (Cox NN) training on signature features
# ============================================================

import numpy as np
import torch
import torchtuples as tt
from sklearn.preprocessing import StandardScaler
from pycox.models import CoxPH
from pycox.evaluation import EvalSurv

np.random.seed(777)
_ = torch.manual_seed(177)

# Prepare training data
# ------------------------------------------------------------

df_train = df_study_train.copy()

# Signature feature columns
cols_standardize = [c for c in df_train.columns if c.startswith("sig_")]

# Standardization
#sig_cols_train = [c for c in df_study_train.columns if c.startswith("sig_")]
scaler = StandardScaler()
x_train = scaler.fit_transform(df_study_train[sig_cols_train]).astype("float32")


# Survival targets
y_train = (
    df_train["time"].values.astype("float32"),
    df_train["event"].values.astype("float32"),
)

# Define Neural Cox model (DeepSurv style)
# ------------------------------------------------------------

in_features = x_train.shape[1]
num_nodes = [64, 32, 16]
out_features = 1
batch_norm = True
dropout = 0.3
output_bias = False

net = tt.practical.MLPVanilla(
    in_features,
    num_nodes,
    out_features,
    batch_norm,
    dropout,
    output_bias=output_bias
)

model = CoxPH(net, tt.optim.Adam)

# Learning rate finder
# ------------------------------------------------------------

batch_size = 256
lrfinder = model.lr_finder(x_train, y_train, batch_size=batch_size, tolerance=10)

best_lr = lrfinder.get_best_lr()
model.optimizer.set_lr(best_lr)

# Train model
# ------------------------------------------------------------

epochs = 512
callbacks = [tt.callbacks.EarlyStopping()]
verbose = True

log = model.fit(
    x_train,
    y_train,
    batch_size=batch_size,
    epochs=epochs,
    callbacks=callbacks,
    verbose=verbose,
    val_data=None  # No leakage: no validation on test sets
)

# Compute baseline hazards
# ------------------------------------------------------------

_ = model.compute_baseline_hazards()

0:	[0s / 0s],		train_loss: 4.7182
1:	[0s / 0s],		train_loss: 4.5573
2:	[0s / 0s],		train_loss: 4.4681
3:	[0s / 0s],		train_loss: 4.4153
4:	[0s / 0s],		train_loss: 4.3403
5:	[0s / 0s],		train_loss: 4.2872
6:	[0s / 0s],		train_loss: 4.2305
7:	[0s / 0s],		train_loss: 4.1755
8:	[0s / 0s],		train_loss: 4.1460
9:	[0s / 0s],		train_loss: 4.1165


In [12]:
# ============================================================
# Evaluation over the 10 independent test sets
# ============================================================

cindex_test_list = []
bs_test_list = []

for i, df_test in enumerate(test_groups):

    # --------------------------------------------------------
    # 1. Apply linear projection (same R_comp as train)
    # --------------------------------------------------------
    df_test = apply_linear_projection(df_test, R_comp, var_embd="embeddings")

    # --------------------------------------------------------
    # 2. Signature extraction
    # --------------------------------------------------------
    _, _, _, _, _, _, df_study_test = prep_signature_cox(df_test, var_struct_seq_list_OG=seq_struc_col)

    # --------------------------------------------------------
    # 3. NO variance threshold here
    # --------------------------------------------------------
    # If you were using variance filtering, it would look like:
    # sig_cols_test = [c for c in df_study_test.columns if c.startswith("sig_")]
    # X_sig_test = selector.transform(df_study_test[sig_cols_test])
    # df_study_test = pd.concat([...], axis=1)
    #
    # But here we keep all signature variables.

    sig_cols_test = [c for c in df_study_test.columns if c.startswith("sig_")]

    # --------------------------------------------------------
    # 4. Standardize with TRAIN scaler only
    # --------------------------------------------------------
    # scaler must have been fitted on train signature columns
    x_test = scaler.transform(
        df_study_test[sig_cols_test]
    ).astype("float32")

    durations_test = df_study_test["time"].values.astype("float32")
    events_test = df_study_test["event"].values.astype("float32")

    # --------------------------------------------------------
    # 5. Survival prediction
    # --------------------------------------------------------
    surv = model.predict_surv_df(x_test)

    # --------------------------------------------------------
    # 6. Evaluation
    # --------------------------------------------------------
    ev = EvalSurv(surv, durations_test, events_test, censor_surv="km")

    c_index = ev.concordance_td()

    t_max = min(3 * 365, durations_test.max())
    time_grid = np.linspace(0.0, t_max, 100)
    ibs = ev.integrated_brier_score(time_grid)

    cindex_test_list.append(c_index)
    bs_test_list.append(ibs)

Timestamps normalized for each patient to the [0, 1] range.
Path dimension: 30
Signature components (order 2): 930
Number of events (deaths): 82 out of 127 (64.57%)
Timestamps normalized for each patient to the [0, 1] range.
Path dimension: 30
Signature components (order 2): 930
Number of events (deaths): 91 out of 127 (71.65%)
Timestamps normalized for each patient to the [0, 1] range.
Path dimension: 30
Signature components (order 2): 930
Number of events (deaths): 89 out of 127 (70.08%)
Timestamps normalized for each patient to the [0, 1] range.
Path dimension: 30
Signature components (order 2): 930
Number of events (deaths): 81 out of 127 (63.78%)
Timestamps normalized for each patient to the [0, 1] range.
Path dimension: 30
Signature components (order 2): 930
Number of events (deaths): 76 out of 126 (60.32%)
Timestamps normalized for each patient to the [0, 1] range.
Path dimension: 30
Signature components (order 2): 930
Number of events (deaths): 78 out of 126 (61.90%)
Timestamps

In [13]:
# ---- Train metric ----

# Unpack y_train
y_time_train_np, y_event_train_np = y_train

# Predict survival on train
pred_surv_train = model.predict_surv_df(x_train)

# Evaluate
ev_train = EvalSurv(
    pred_surv_train,
    y_time_train_np,
    y_event_train_np,
    censor_surv='km'
)


c_index_train = ev_train.concordance_td()

# ============================================================
# Final metrics
# ============================================================

mean_cindex = np.mean(cindex_test_list)
sd_cindex = np.std(cindex_test_list, ddof=1)
lower_c, upper_c = jackknife_confidence_interval(cindex_test_list)

mean_bs = np.mean(bs_test_list)
sd_bs = np.std(bs_test_list, ddof=1)
lower_bs, upper_bs = jackknife_confidence_interval(bs_test_list)
print(
    f"C-index Train: {c_index_train:.3f}"
)

print("\n -------- Test --------")
print(
    f"Mean Test C-index: {mean_cindex:.3f} "
    f"(sd {sd_cindex:.3f}) "
    f"[{lower_c:.3f}, {upper_c:.3f}]"
)

print(
    f"Mean Test IBS (0–3y): {mean_bs:.3f} "
    f"(sd {sd_bs:.3f}) "
    f"[{lower_bs:.3f}, {upper_bs:.3f}]"
)

C-index Train: 0.458

 -------- Test --------
Mean Test C-index: 0.454 (sd 0.031) [0.435, 0.473]
Mean Test IBS (0–3y): 0.187 (sd 0.018) [0.176, 0.198]
